<br>
<br>
<br>
<br>
<br>
<br>
<br>
<center>
    <h1 style = "font-size:48px; font-weight:normal"><b>AI</b>-TECH</style></h1>
    <br>
<br>
<br>
<br>
    <h1 style = "font-size:24px; font-weight:normal">LABORATORIUM</style></h1>
    <h1 style = "font-size:24px">Przetwarzanie Multimediów w
Systemach Decyzyjnych
</style></h1>
    <h1 style = "font-size:24px">Detekcja obiektów</style></h1>
    <h1 style = "font-size:24px">Sebastian Cygert</style></h1>
    
   
</center>
<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>
    </body>
    </html>

![stopka.svg](stopka.svg)

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

# 1. Uruchomienie modelu detekcji obiektów

In [ ]:
# Instalacja biblioteki detectron2 - służącej do rozpoznawania obrazu
# Biblioteka: https://github.com/facebookresearch/detectron2
# Instalacja: https://detectron2.readthedocs.io/tutorials/install.html

In [ ]:
# Instalacja biblioteki wspomagającej pracę ze zbiorem danych COCO (obliczenie metryk, pobranie części zbioru, wizualizacja wyników)
!pip install fiftyone
!pip install fiftyone-db-ubuntu2204


In [ ]:
!python -m pip install pyyaml==5.1

import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities.
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

In [ ]:
#Import przydatnych pakietów
from detectron2.utils.logger import setup_logger
setup_logger()

import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

Pobranie przykładowego obrazu ze zbioru MS-COCO (https://cocodataset.org/#home).

In [ ]:
!wget http://images.cocodataset.org/val2017/000000006471.jpg -q -O input_image.jpg
img = cv2.imread("./input_image.jpg")
cv2_imshow(img)

W kolejnym kroku zostanie utworzony odpowiedni plik konfiguracyjny, który posłuży nam do detekcji obiektów.
W tym celu wykorzystany zostanie jeden w wcześniej wytrenowanych modeli dostępnych w "model-zoo" biblioteki detectron2: (https://github.com/facebookresearch/detectron2/blob/main/MODEL_ZOO.md)

In [ ]:
cfg = get_cfg()
model_name = "COCO-Detection/faster_rcnn_R_50_FPN_1x.yaml" #plik konfiguracyjny
cfg.merge_from_file(model_zoo.get_config_file(model_name))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.0  # próg detekcji
# Pobranie wytrenowanego wcześniej modelu
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(model_name)

In [ ]:
# Utworzenie modelu predykcji
predictor2 = DefaultPredictor(cfg)
# Predykcja
outputs = predictor2(img)

# Format danych wyjściowych https://detectron2.readthedocs.io/en/latest/tutorials/models.html#model-output-format
instances = outputs["instances"]
THRESH = 0.2
# Filtracja predykcji z niskim poziomem pewności.
confident_detections = instances[instances.scores > THRESH]
# Wypisanie danych wyjściowych z modelu. Przyjrzyj się formatowi danych i definicjom pól
print("scores", confident_detections.scores)
print("classes", confident_detections.pred_classes)
print("pred_boxes", confident_detections.pred_boxes)

In [ ]:
# Wizulizacja predykcji
v = Visualizer(img[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.3)
out = v.draw_instance_predictions(confident_detections.to("cpu"))
cv2_imshow(out.get_image()[:, :, ::-1])

#Zadanie 1.

Przetestuj model dla różnych wartości parametru "Thresh". Którą wartość uważasz za optymalną? W sprawozdaniu podaj wartość i wrzuć obraz z detekcjami modelu.

# 2. Obliczenie dokładności modelu na zbiorze COCO
W tym celu pobierzemy niewielki podzbiór walidacyjny (100 zdjęć) i obliczymy skuteczność modelu na tym podzbiorze dla wybranych klas

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
#Pobranie 150 losowych próbek ze zbioru COCO
dataset = foz.load_zoo_dataset(
  "coco-2017",
  split="validation",
  label_types=["detections"],
  max_samples=150,
)
dataset.persistent = True

# Uruchomienie predykcji na części zbioru COCO.
Odczytaj czas inferencji modelu.

In [ ]:
RANDOM_SEED = 5#TODO - wpisz tutaj losową wartość służącą do inicjalizacji zbioru
predictions_view = dataset.take(100, seed=RANDOM_SEED)
classes = dataset.default_classes
print("Klasy", classes)
with fo.ProgressBar() as pb:
    for sample in pb(predictions_view):
        #pobierz obraz
        image = cv2.imread(sample.filepath)

        height, width, channels = image.shape

        #predykcja
        preds = predictor2(image)
        labels = preds["instances"].pred_classes
        scores = preds["instances"].scores
        boxes = preds["instances"].pred_boxes

        # Konwersja detekcji do formatu biblioteki fiftyOne
        detections = []
        for label, score, box in zip(labels, scores, boxes):
            x1, y1, x2, y2 = box

            rel_box = [x1 / width, y1 / height, (x2 - x1) / width, (y2 - y1) / height]
            detections.append(
                fo.Detection(
                    label=classes[label+1],
                    bounding_box=rel_box,
                    confidence=score
                )
            )
      # Save predictions to dataset
        sample["faster_rcnn"] = fo.Detections(detections=detections)
        sample.save()

print("Predykcje ukończone. Odczytaj czas inferencji (linijkę wyżej)")

Obliczenie metryk dokładności

In [ ]:
from fiftyone import ViewField as F
# Odrzucenie predykcji o niskiej pewności
high_conf_view = predictions_view.filter_labels("faster_rcnn", F("confidence") > 0.25)
results = high_conf_view.evaluate_detections(
    "faster_rcnn",
    gt_field="ground_truth",
    eval_key="eval",
    compute_mAP=True,
)
counts = dataset.count_values("ground_truth.detections.label")
# Obliczenie metryk prezycji (precision) i przywołania (recall) dla 5. najbardziej popularnych klas w naszym podzbiorze danych
classes_top5 = sorted(counts, key=counts.get, reverse=True)[:5]
results.print_report(classes=classes_top5)

In [ ]:
# Wizulacji krzywej precyzji i odwołania dla każdej z klasy. Pole pod wykresem oznacza metrykę average precision
plot = results.plot_pr_curves(classes=["person", "car"])
plot.show()

In [ ]:
# Wizualizacja predykcji
session = fo.launch_app(predictions_view)

#Zadanie 2.

Zapisz w tabelce czas działania modelu (FPS) oraz dokładność (Average Precision - AP) dla klas "person", "car" oraz dwóch wybranych licznych klas z Twojego podzbioru.

W sprawozdaniu zapisz Twoje uwagi dotyczące dokładności modelu dla różnych klas.

#Zadanie 3.

Uzupełnij tabelkę z zadania 2 o wyniki dla dwóch wybranych modeli dostępnych w bibliotece detectron2 (interesuje nas sekcja COCO-detection baselines: https://github.com/facebookresearch/detectron2/blob/main/MODEL_ZOO.md#coco-object-detection-baselines)

Wybierz zróznicowane modele (np. jeden model charakteryzuje się dużą dokładnością a inny model charakteryzuje się szybkością działania).

W sprawozdaniu zapisz swój komentarz odnoście dokładności i czasu działania różnych modeli, czy występuje tu jakaś zależność?

UWAGA. Niektóre modele mogą generować błędy (brak kompatybilności API), w takim przypadku należy wybrać inny model.

#3. Wrażliwość modeli na szum wejściowy

Współczesne modele uczenia maszynowego (w tym rozpoznawania obrazu) są znane z tego, że ich dokładność mocno spada gdy warunki testowe różnią się od tych treningowych. W tej części ćwiczenia skupimy się na wrażliwości model na szum wejściowy.

W tym celu do testowanych obrazów dodamy niewielki szum gaussowski i sprawdzimy jak taki zabieg wpływa na dokładność modelu.

Przetestuj wpływ wartości wariancji (*variance*) na obraz końcowy.

In [ ]:
# Definicja funkcji dodającej szum gaussowski
def add_noise(input_img, var = 0.0):
    mean = 0.0
    sigma = var**0.5
    gauss = np.random.normal(mean,sigma,input_img.size)
    gauss = gauss.reshape(input_img.shape[0], input_img.shape[1], input_img.shape[2]).astype('uint8')
    # Dodaj szum do obrazu
    img_gauss = cv2.add(input_img,gauss)
    return img_gauss

variance = 0.0
img_gauss = add_noise(img, variance)
cv2_imshow(img)
cv2_imshow(img_gauss)

# Zadanie 4.
Wybierz dwie różne wartości wariancji. Sprawdź jak dodadnie takiego szumu w czasie inferencji wpływa na metryki Average Precision. W tym celu wybierz jeden z modeli wykorzystanych w ćwiczeniu nr 3 oraz te same klasy, które opisałeś w ćwiczeniu nr 3.

W tym celu wystarczy dodać jedną linijkę kodu dodającą szum do kodu znajdującego się w bloku (Uruchomienie predykcji na części zbioru COCO) https://colab.research.google.com/drive/15vuDB-AbNDZmqW4TfwI_02AYbl4EH7Aa#scrollTo=NcB4_r4jVm3j&line=3&uniqifier=1.

# 4. Uruchomienie predykcji na obrazach video

W tej części ćwiczenia skupimy się na uruchomieniu modeli detekcji obiektów na obrazach video. W tym celu wykorzystane zostaną dwa nagrania: jedno nagranie będzie nagraniem o wysokiej jakości, natomiast drugie nagranie charakteryzować się będzie niską jakością obrazu, słabymy warunkami oświetleniowymi i nietypowym kątem nagrania.

In [ ]:
# instalacja biblioteki do pobierań nagrań z youtube
!pip install youtube-dl
# pobranie kodu biblioteki aby uruchomić pliku demo.py, który ułatwia inferencję na wielu plikach
!git clone https://github.com/facebookresearch/detectron2

In [ ]:
# pobranie nagrania
!youtube-dl https://www.youtube.com/watch?v=Y1jTEyb3wiI -f 18 -o video.mp4
# wycięcie 5 sekundowej części nagrania
!ffmpeg -i video.mp4 -t 00:00:05 -c:v copy video-clip.mp4

In [ ]:
# uruchomienie inferencji na obrazie video.
# zwróć uwagę na argumenty funkcji
%run detectron2/demo/demo.py --config-file detectron2/configs/COCO-Detection/faster_rcnn_R_50_FPN_1x.yaml --video-input video-clip.mp4 --confidence-threshold 0.4 --output video-output.mkv \
  --opts MODEL.WEIGHTS https://dl.fbaipublicfiles.com/detectron2/COCO-Detection/faster_rcnn_R_50_FPN_1x/137257794/model_final_b275ba.pkl


In [ ]:
# Pobranie wyników
from google.colab import files
files.download('video-output.mkv')

# Zadanie 5

Samodzielnie dokonaj infernecji na nagraniu dostępnym pod adresem: https://youtu.be/Xw-Ux_NH58Q
Zwróć uwagę aby ustawić inne nazwy plików przy pobieraniu nagrania, jego wycinaniu i samej inferencji.

W sprawozdaniu porównaj dokładność modeli na obu z tych nagrań.

# Zadanie 6.

Wykonaj inferencję dla drugiego z nagrań wykorzystując najbardziej dokładny model z model-zoo. Czy zauważyłeś poprawę?
Co można by zrobić aby poprawić dokładność detekcji obiektów w takim przypadku?

# 5. Detekcja punktów charakterystycznych

Biblioteka detectron2 zawiera wiele różnych wbudowanych modeli. Jednym z przykładówych jest estymacja sylwetki ludzkiej.

In [ ]:
# Inference with a keypoint detection model
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml")
predictor = DefaultPredictor(cfg)
outputs = predictor(img)
v = Visualizer(img[:,:,::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
cv2_imshow(out.get_image()[:, :, ::-1])